In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from methyldl.data.soft_labeling import (
    extract_cpg_signature,
    signature_distance,
    apply_normalized_knn_smoothing,
)
from EDA.edautils import plot_soft_label_distribution

In [2]:
from torch.nn import Softmax

In [3]:
import os

In [4]:
reference_genome = "hg19"

In [ ]:
data_path = f"/home/luna.kuleuven.be/u0169940/Data/Loyfer/SimulatedReads_205files_{reference_genome}/all_reads_transformed_uxm_prepared.parquet"
all_data = pd.read_parquet(data_path)
all_data = all_data[all_data["NCPGS"] >= 4]

In [ ]:
# Using tqdm for pandas apply (optional, but helpful for large datasets)
tqdm.pandas(desc="Extracting Signatures")
all_data["cpg_sig"] = all_data.progress_apply(extract_cpg_signature, axis=1)

In [ ]:
df = all_data.copy()
label_col = "original_label"
num_classes = 39

In [ ]:
all_data["original_label"] = all_data["original_label"].apply(lambda x: np.int32(x))
all_data["cpg_sig"] = all_data["cpg_sig"].apply(tuple)

In [ ]:
base_counts = df.groupby(["name", "cpg_sig", label_col]).size().unstack(fill_value=0)
base_counts = base_counts.reindex(columns=range(num_classes), fill_value=0)
base_counts = base_counts.reset_index()
base_counts["total_reads"] = base_counts[list(range(num_classes))].sum(axis=1)

In [ ]:
mapping_df = apply_normalized_knn_smoothing(
    base_counts, min_reads=30, num_classes=39, max_distance=0.41
)

In [ ]:
final_data = all_data.merge(mapping_df, on=["name", "cpg_sig"], how="left")

In [ ]:
ctypes = all_data["dmr_ctype"].unique()

In [ ]:
ctypes_dict = {key: value for (key, value) in zip(sorted(ctypes), range(len(ctypes)))}

In [ ]:
final_data["dmr_ctype_label"] = final_data["dmr_ctype"].apply(lambda x: ctypes_dict[x])

In [ ]:
labels_dict = {
    x[1]: x[0]
    for x in final_data[["dmr_ctype", "dmr_ctype_label"]]
    .drop_duplicates()
    .to_dict(orient="split")["data"]
}

In [ ]:
import ast
from matplotlib.patches import Patch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [5]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


def split_by_file_and_class(
    df,
    target_col="original_label",
    file_col="file",
    train_ratio=0.7,
    valid_ratio=0.15,
    test_ratio=0.15,
    random_state=42,
):
    """
    Splits the dataset into train, valid, and test sets.
    Prioritizes file-level isolation to prevent data leakage,
    while EXPLICITLY guaranteeing every class is represented in all splits.
    """
    assert np.isclose(
        train_ratio + valid_ratio + test_ratio, 1.0
    ), "Ratios must sum to 1.0"

    train_chunks, valid_chunks, test_chunks = [], [], []

    # Group the entire dataset by the ground truth cell type
    grouped = df.groupby(target_col)

    for label, group in grouped:
        # Get read counts per file for this specific cell type
        file_counts = group[file_col].value_counts().to_dict()
        files_sorted = sorted(
            file_counts.keys(), key=lambda k: file_counts[k], reverse=True
        )
        n_files = len(files_sorted)
        total_reads = sum(file_counts.values())

        if n_files >= 3:
            # REVISED GREEDY FILE-LEVEL ALLOCATION WITH COVERAGE OVERRIDE
            targets = {
                "train": total_reads * train_ratio,
                "valid": total_reads * valid_ratio,
                "test": total_reads * test_ratio,
            }
            current_counts = {"train": 0, "valid": 0, "test": 0}
            allocations = {"train": [], "valid": [], "test": []}

            for i, file in enumerate(files_sorted):
                f_count = file_counts[file]
                remaining_files = n_files - i

                # Check which buckets still have 0 files
                empty_buckets = [b for b in targets.keys() if not allocations[b]]

                # FORCE COVERAGE: If we only have just enough files left to fill the empty
                # buckets, we must abandon the ratio math and fill the empty buckets.
                if len(empty_buckets) >= remaining_files:
                    best_bucket = max(empty_buckets, key=lambda b: targets[b])
                else:
                    # Otherwise, use standard greedy: give to bucket with highest deficit
                    deficits = {k: targets[k] - current_counts[k] for k in targets}
                    best_bucket = max(deficits, key=deficits.get)

                allocations[best_bucket].append(file)
                current_counts[best_bucket] += f_count

            train_chunks.append(group[group[file_col].isin(allocations["train"])])
            valid_chunks.append(group[group[file_col].isin(allocations["valid"])])
            test_chunks.append(group[group[file_col].isin(allocations["test"])])

        elif n_files == 2:
            # 2 FILES: Keep Train isolated. Split File 2 into Valid/Test.
            file1, file2 = files_sorted[0], files_sorted[1]

            # Put the largest file in Train
            train_chunks.append(group[group[file_col] == file1])

            # Split the second file between valid and test
            file2_df = group[group[file_col] == file2]
            val_prop = valid_ratio / (valid_ratio + test_ratio)

            val_df, test_df = train_test_split(
                file2_df, train_size=val_prop, random_state=random_state
            )
            valid_chunks.append(val_df)
            test_chunks.append(test_df)

        else:
            # 1 FILE: Unavoidable read-level split to ensure class coverage.
            file_df = group
            train_df, temp_df = train_test_split(
                file_df, train_size=train_ratio, random_state=random_state
            )

            val_prop = valid_ratio / (valid_ratio + test_ratio)
            val_df, test_df = train_test_split(
                temp_df, train_size=val_prop, random_state=random_state
            )

            train_chunks.append(train_df)
            valid_chunks.append(val_df)
            test_chunks.append(test_df)

    # Combine all the chunks
    df_train = pd.concat(train_chunks, ignore_index=True)
    df_valid = pd.concat(valid_chunks, ignore_index=True)
    df_test = pd.concat(test_chunks, ignore_index=True)

    return df_train, df_valid, df_test

In [ ]:
train_df, valid_df, test_df = split_by_file_and_class(
    final_data, target_col="original_label"
)

In [ ]:
plot_soft_label_distribution(
    train_df, target_dmr_ctype="Colon-Fibro", target_label=12, labels_dict=labels_dict
)

In [ ]:
new_data_path = "/home/luna.kuleuven.be/u0169940/Data/Loyfer/SoftLabelsTrainingData_205files_pooled_Jaccard_hg38_mincpg_4_minlen_10/"

In [ ]:
# os.mkdir(new_data_path)
for name, df in zip(["train", "valid", "test"], [train_df, valid_df, test_df]):
    df.to_parquet(os.path.join(new_data_path, f"{name}.parquet"), index=False)

In [ ]:
import pandas as pd

In [ ]:
train, valid, test = [
    pd.read_parquet(new_data_path + x + ".parquet") for x in ["train", "valid", "test"]
]

In [ ]:
x = len(train), len(valid), len(test)

In [ ]:
x[0] / sum(x), x[1] / sum(x), x[2] / sum(x)

In [ ]:
train_files = set(train["file"])
valid_files = set(valid["file"])
test_files = set(test["file"])

In [ ]:
(valid_files).intersection(test_files)

### Soft labels without leaking

In [ ]:
from tqdm import tqdm

In [ ]:
# data_path = "/home/luna.kuleuven.be/u0169940/Data/Loyfer/SoftLabelsTrainingData_205files_NO_POOLING_Jaccard_hg38_mincpg_4_minlen_10_no_data_leak/"
data_path = "/home/luna.kuleuven.be/u0169940/Data/Loyfer/SoftLabelsForRRBSsplits_205files_pooled_Jaccard_hg38_mincpg_4_minlen_10_d041/"

In [ ]:
# train, valid, test = [
#     pd.read_parquet(data_path + x + ".parquet") for x in ["train", "valid", "test"]
# ]

train, valid = [pd.read_parquet(data_path + x + ".parquet") for x in ["train", "valid"]]

In [26]:
WITH_POOLING = False

In [27]:
if WITH_POOLING:
    max_distance = 0.41
    min_reads = 30
    new_data_path = "/home/luna.kuleuven.be/u0169940/Data/Loyfer/SoftLabelsTrainingData_205files_pooled_Jaccard_hg38_mincpg_4_minlen_10_no_data_leak_d041/"
else:
    max_distance = 0
    min_reads = 1
    new_data_path = "/home/luna.kuleuven.be/u0169940/Data/Loyfer/SoftLabelsTrainingData_205files_NO_POOLING_Jaccard_hg38_mincpg_4_minlen_10_no_data_leak/"

In [8]:
def process_by_split(df):
    tqdm.pandas(desc="Extracting Signatures")
    df["cpg_sig"] = df.progress_apply(extract_cpg_signature, axis=1)
    df.drop(
        [
            "normalized_counts",
            "soft_label",
            "raw_reads_pooled",
            "total_normalized_reads",
            "num_signatures_pooled",
        ],
        axis=1,
        inplace=True,
    )
    label_col = "original_label"
    num_classes = 39
    base_counts = (
        df.groupby(["name", "cpg_sig", label_col]).size().unstack(fill_value=0)
    )
    base_counts = base_counts.reindex(columns=range(num_classes), fill_value=0)
    base_counts = base_counts.reset_index()
    base_counts["total_reads"] = base_counts[list(range(num_classes))].sum(axis=1)
    mapping_df = apply_normalized_knn_smoothing(
        base_counts, min_reads=min_reads, num_classes=39, max_distance=max_distance
    )
    final_data = df.merge(mapping_df, on=["name", "cpg_sig"], how="left")
    return final_data

In [ ]:
train = process_by_split(train)
valid = process_by_split(valid)
test = process_by_split(test)

import os

os.mkdir(new_data_path)
for name, df in zip(["train", "valid", "test"], [train, valid, test]):
    df.to_parquet(os.path.join(new_data_path, f"{name}.parquet"), index=False)

### Process for RRBS

In [ ]:
train["split"] = "train"
valid["split"] = "valid"
df = pd.concat([train.copy(True), valid.copy(True)], axis=0)

In [ ]:
df = process_by_split(df)
train = df[df["split"] == "train"]
train.drop(["split"], axis=1, inplace=True)
valid = df[df["split"] == "valid"]
valid.drop(["split"], axis=1, inplace=True)
new_data_path = "/home/luna.kuleuven.be/u0169940/Data/Loyfer/SoftLabelsForRRBSsplits_205files_NO_POOLING_Jaccard_hg38_mincpg_4_minlen_10_d041/"

In [ ]:
os.mkdir(new_data_path)
for name, df in zip(["train", "valid"], [train, valid]):
    df.to_parquet(os.path.join(new_data_path, f"{name}.parquet"), index=False)

### Fitting Soft Labels Classifier

In [ ]:
if reference_genome == "hg19":
    data_path = f"/home/luna.kuleuven.be/u0169940/Data/Loyfer/SimulatedReads_205files_{reference_genome}/all_reads_transformed_uxm_prepared.parquet"
    all_data = pd.read_parquet(data_path)
    # # all_data["NCPGS"] = all_data["original_methyl"].apply(lambda x: x.count("C")+x.count("T"))
    all_data = all_data[all_data["NCPGS"] >= 4]
    train, train2, valid = split_by_file_and_class(
        all_data, target_col="original_label"
    )
    train = pd.concat([train, train2], axis=0)

    new_data_path = "/home/luna.kuleuven.be/u0169940/Data/Loyfer/HG19_ForRRBSsplits_LOOKUPONLY_205files/"
    # os.mkdir(new_data_path)
    for name, df in zip(["train", "valid"], [train, valid]):
        df.to_parquet(os.path.join(new_data_path, f"{name}.parquet"), index=False)
    # train, valid = [pd.read_parquet(new_data_path + x + ".parquet") for x in ["train", "valid"]]

In [37]:
pd.read_parquet(new_data_path + "train.parquet").shape

(2910114, 29)

In [11]:
all_data["name"].unique().shape

(816,)

In [14]:
import numpy as np

In [15]:
RRBS = True

In [16]:
from methyldl.modelling.classifiers.lookup import LookupClassifier, SoftLabelConfig

In [28]:
lookup_config = SoftLabelConfig(
    num_classes=39,
    label_col="original_label",
    min_reads=min_reads,
    max_distance=max_distance,
)

In [29]:
classifier = LookupClassifier(lookup_config)

In [19]:
if RRBS:
    df = pd.concat([train.copy(True), valid.copy(True)], axis=0)
else:
    df = train.copy(True)

In [ ]:
df.drop(
    [
        "normalized_counts",
        "soft_label",
        "raw_reads_pooled",
        "total_normalized_reads",
        "num_signatures_pooled",
        "cpg_sig",
    ],
    axis=1,
    inplace=True,
)

In [30]:
classifier.fit(df)

Smoothing Regions: 100%|██████████| 814/814 [01:42<00:00,  7.91it/s]


In [21]:
reference_genome.capitalize()

'Hg19'

In [31]:
if WITH_POOLING:
    name = "LookupClassifier_SoftLabels_Ours"
else:
    name = "LookupClassifier_SoftLabels_WITHOUTPOOLING"

In [32]:
classifier.save(
    f"/data/dmytro/Experiments/cfsort_rrbs/{reference_genome.capitalize()}_{name}/lookup_fitted.joblib"
)

### Cannonical Soft Labels

In [ ]:
alpha = 0.1

In [ ]:
def construct_canonical_soft_labels(target, epsilon, n_labels):
    uniform_vector = np.array([1 / n_labels] * n_labels)
    target_vector = np.zeros(n_labels)
    target_vector[target] = 1
    return target_vector * (1 - epsilon) + uniform_vector * epsilon

In [ ]:
for df in [train, valid]:
    df["soft_label"] = df["label"].apply(
        lambda x: construct_canonical_soft_labels(x, 0.1, 40)
    )

In [ ]:
canonical_path = "/home/luna.kuleuven.be/u0169940/Data/Loyfer/CanonicalSoftWithBckgForRRBSsplits_epsilon01_TrainingData_205files_hg38_mincpg_4_minlen_10/"
os.mkdir(canonical_path)
for name, df in zip(["train", "valid"], [train, valid]):
    df.to_parquet(os.path.join(canonical_path, f"{name}.parquet"), index=False)